In [33]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import torch
import torch.nn as nn
import numpy as np
import pickle
import time
from tqdm import tqdm
from utils import Data_Test
from model import create_model_Fave, Att_Fave_model
from scipy import stats

DATASET = 'amazon-beauty'  # change this to 'ml-100k','steam' or 'amazon-beauty' when needed
DATASET_CONFIG = {
    'ml-100k': {
        'data_path': '../datasets/data/ml-100k/dataset.pkl',
        'ckpt_path': '../best for ml-100k/pretrain_best_finetuned.pt',
    },
    'amazon-beauty': {
        'data_path': '../datasets/data/amazon_beauty/dataset.pkl',
        'ckpt_path': '../best for amazon_beauty/pretrain_best_finetuned.pt',
    },
    'steam': {
        'data_path': '../datasets/data/steam/dataset.pkl',
        'ckpt_path': '../best for steam/pretrain_best_finetuned.pt',
    },
}

def get_dataset_config(dataset_name):
    if dataset_name not in DATASET_CONFIG:
        raise ValueError(f'Unsupported dataset: {dataset_name}. Available: {list(DATASET_CONFIG.keys())}')
    return DATASET_CONFIG[dataset_name]

class Config:
    def __init__(self, dataset_name=DATASET):
        dataset_config = get_dataset_config(dataset_name)
        self.log_file = 'log/'
        self.random_seed = 1997
        self.device = 'cuda'
        self.num_gpu = 1
        self.batch_size = 512          
        self.max_len = 50
        self.metric_ks = [10, 20]    
        self.hidden_size = 128
        self.num_blocks = 4
        self.last = 2                 

        self.dropout = 0.3
        self.emb_dropout = 0.3
        self.drop_path_rate = 0.2
        self.mask_ratio = 1.0           

        self.dataset = dataset_name
        self.eps = 0.001
        self.lambda_uncertainty = 0.001

        self.sample_N = 1            
        self.eps_reverse = 0.001   
             
        self.mask_end = 0.5  
        self.gamma_init = 1e-2
        self.train_mask_ratio=0.5
        self.infer_mask_ratio=0.6
        self.loss_aux_weight=0.5
        self.loss_straight_weight=0.1
        self.loss_pretrain_weight=0.3
        
        self.m_logNorm = 1.0
        self.s_logNorm = 0.6
        self.s_modsamp = 1.0
        self.sampling_method = 'mode'
        

        self.ckpt_path = dataset_config['ckpt_path']

        self.item_num = 0
        self.user_num = 0
        self.vis_samples=50

# Load configuration
args = Config(DATASET)
print(f"Inference configuration loaded. Dataset: {args.dataset}, Checkpoint: {args.ckpt_path}")

Inference configuration loaded. Dataset: amazon-beauty, Checkpoint: ../best for amazon_beauty/pretrain_best_finetuned.pt


In [34]:
def cal_hr(label, predict, ks):
    max_ks = max(ks)
    _, topk_predict = torch.topk(predict, k=max_ks, dim=-1)
    hit = label == topk_predict
    hr = [hit[:, :ks[i]].sum().item()/label.size()[0] for i in range(len(ks))]
    return hr

def dcg(hit):
    log2 = torch.log2(torch.arange(1, hit.size()[-1] + 1) + 1).unsqueeze(0)
    rel = (hit/log2).sum(dim=-1)
    return rel

def cal_ndcg(label, predict, ks):
    max_ks = max(ks)
    _, topk_predict = torch.topk(predict, k=max_ks, dim=-1)
    hit = (label == topk_predict).int()
    ndcg = []
    for k in ks:
        max_dcg = dcg(torch.tensor([1] + [0] * (k-1)))
        predict_dcg = dcg(hit[:, :k])
        ndcg.append((predict_dcg/max_dcg).mean().item())
    return ndcg

def hrs_and_ndcgs_k(scores, labels, ks):
    metrics = {}
    # Move tensors to CPU for metric computation and lower GPU memory usage
    scores_cpu = scores.detach().cpu()
    labels_cpu = labels.detach().cpu()
    
    ndcg = cal_ndcg(labels_cpu, scores_cpu, ks)
    hr = cal_hr(labels_cpu, scores_cpu, ks)
    
    for k, ndcg_temp, hr_temp in zip(ks, ndcg, hr):
        metrics[f'HR@{k}'] = hr_temp
        metrics[f'NDCG@{k}'] = ndcg_temp
    return metrics

In [35]:
dataset_config = get_dataset_config(args.dataset)
path_data = dataset_config['data_path']
if not os.path.exists(path_data):
    raise FileNotFoundError(f"Data file not found: {path_data}")

print(f"Loading data: {path_data} ...")
with open(path_data, 'rb') as f:
    data_raw = pickle.load(f)

args.item_num = len(data_raw['smap']) + 1
args.user_num = len(data_raw['train']) + 1
print(f"Data loaded. Item Num: {args.item_num}, User Num: {args.user_num}")

test_data = Data_Test(data_raw['train'], data_raw['val'], data_raw['test'], args)
test_data_loader = test_data.get_pytorch_dataloaders()
print(f"Test DataLoader ready. Batch count: {len(test_data_loader)}")

print("Initializing model...")
FM_rec = create_model_Fave(args)
model = Att_Fave_model(FM_rec, args).to(args.device)

if os.path.exists(args.ckpt_path):
    print(f"Loading checkpoint: {args.ckpt_path}")
    checkpoint = torch.load(args.ckpt_path, map_location=args.device)
    
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint
        
    new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    
    msg = model.load_state_dict(new_state_dict, strict=False)
    print(f"Weights loaded successfully. Status: {msg}")
else:
    raise FileNotFoundError(f"Checkpoint not found: {args.ckpt_path}")

model.stage = 'finetune'  # pretrain, finetune
model.eval()
print(f"Model is ready. Current stage: {model.stage}")


Loading data: ../datasets/data/amazon_beauty/dataset.pkl ...
Data loaded. Item Num: 12102, User Num: 22364
Test DataLoader ready. Batch count: 44
Initializing model...
Loading checkpoint: ../best for amazon_beauty/pretrain_best_finetuned.pt
Weights loaded successfully. Status: <All keys matched successfully>
Model is ready. Current stage: finetune


In [36]:
import torch
import numpy as np
from tqdm import tqdm
import os

print(f"Starting inference on {args.device}...")

# ==================================================================================
# Core evaluation flow (recommendation metrics only)
# ==================================================================================

# 1. Initialize the metric dictionary
test_metrics_dict = {f'HR@{k}': [] for k in args.metric_ks}
test_metrics_dict.update({f'NDCG@{k}': [] for k in args.metric_ks})

# 2. Switch model to evaluation mode
model.eval()

# 3. Start the test loop
with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(test_data_loader, desc="Testing")):
        # Move batch to device
        seqs, labels = batch
        seqs = seqs.to(args.device)
        labels = labels.to(args.device)
        
        # Forward pass (train_flag=False)
        # Pass arguments according to the model definition
        _, rep_fave, _, _, _, _ = model(seqs, labels, train_flag=False)
        
        # Compute recommendation scores
        scores_rec_fave = model.fave_rep_pre(rep_fave)

        # Compute metrics (HR, NDCG)
        metrics = hrs_and_ndcgs_k(scores_rec_fave, labels, args.metric_ks)
        
        # Collect results
        for k, v in metrics.items():
            test_metrics_dict[k].append(v)

# ==================================================================================
# Final result printing
# ==================================================================================
print("\n" + "="*60)
print(f"Final evaluation report (Checkpoint: {os.path.basename(args.ckpt_path)})")
print("="*60)

print("Recommendation metrics:")
for key, values in test_metrics_dict.items():
    # Compute the mean and convert it to a percentage
    mean_val = round(np.mean(values) * 100, 4)
    print(f"   - {key:<10}: {mean_val}")

print("="*60)

Starting inference on cuda...


Testing: 100%|██████████| 44/44 [00:02<00:00, 21.91it/s]


Final evaluation report (Checkpoint: pretrain_best_finetuned.pt)
Recommendation metrics:
   - HR@10     : 8.4657
   - HR@20     : 11.8693
   - NDCG@10   : 4.9673
   - NDCG@20   : 5.8239
